# Validación del Simulador FAR(1) — versión corregida

Notebook que verifica el módulo `simulation_pipeline_linear.py` corregido.

| # | Problema original | Corrección |
|---|---|---|
| 1 | `gaussian_filter1d` reducía `std(ε_t)` a ≈37% | Renormalización post-filtrado |
| 2 | Tendencias `linear`/`bump` dominaban el proceso para `n_curves` grande | Normalizadas por `n_curves` |
| 3 | `burn_in=50` apenas suficiente | `burn_in=100` por defecto |
| 4 | Sin validaciones de entrada | Checks de shape, tipo, `noise_std>0` |
| 5 | Sin diagnóstico integrado | `summary_stats()` y `plot_acf()` |


## 0. Imports y configuración


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, normaltest
from scipy import stats

from simulation_pipeline_linear import (
    FunctionalDomain,
    FARSimulator,
    gaussian_integral_kernel,
    build_integral_matrix,
)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

domain = FunctionalDomain.regular(s_min=0, s_max=1, n_points=100)
Psi    = build_integral_matrix(domain, gaussian_integral_kernel,
                               bandwidth=0.05, decay=0.6)
print('Dominio:', domain)


## 1. Corrección 1 — Calibración del ruido suavizado

`gaussian_filter1d(sigma=2)` reduce la desviación estándar del vector de
ruido en un factor ≈ 0.37. En la versión original, `noise_std=1.0` producía
`std(ε_t) ≈ 0.37`. Ahora el vector se renormaliza exactamente.


In [ ]:
from scipy.ndimage import gaussian_filter1d

rng = np.random.default_rng(42)
noise_std = 1.0
stds_before, stds_after = [], []

for _ in range(2000):
    z  = rng.normal(0, noise_std, 100)
    zs = gaussian_filter1d(z, sigma=2.0)
    stds_before.append(zs.std())
    stds_after.append((zs * noise_std / zs.std()).std())

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(stds_before, bins=40, color='steelblue', alpha=0.8)
axes[0].axvline(noise_std, color='crimson', lw=2, label=f'objetivo={noise_std}')
axes[0].set_title(f'Sin corrección: std(e) ≈ {np.mean(stds_before):.3f}')
axes[0].legend()
axes[1].hist(stds_after, bins=40, color='seagreen', alpha=0.8)
axes[1].axvline(noise_std, color='crimson', lw=2, label=f'objetivo={noise_std}')
axes[1].set_title(f'Con corrección: std(e) = {np.mean(stds_after):.4f}')
axes[1].legend()
plt.suptitle('Distribución de std(e_t) — 2000 muestras', fontsize=13)
plt.tight_layout(); plt.show()


## 2. Corrección 2 — Tendencias normalizadas

Las tendencias ahora escalan en `t / n_curves ∈ [0,1]`, de forma que la
amplitud máxima es independiente de cuántas curvas se generen.


In [ ]:
trends = ['zero', 'linear', 'sinusoidal', 'bump']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, trend_name in zip(axes, trends):
    sim = FARSimulator(domain=domain, n_curves=100, Psi=Psi,
                       trend=trend_name, noise_std=0.3,
                       noise_type='smooth', burn_in=100, random_state=42)
    X = sim.simulate()
    for t in range(0, 100, 3):
        ax.plot(domain.grid, X[t], alpha=0.25, lw=0.7, color='steelblue')
    ax.plot(domain.grid, X.mean(axis=0), color='crimson', lw=2, label='media')
    ax.set_title(f'trend={trend_name}')
    ax.set_xlabel('s'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('FAR(1) — cuatro tendencias (escala homogénea)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


## 3. Estabilidad del operador Psi

Condición necesaria de estacionariedad: `|lambda|_max < 1`.


In [ ]:
bandwidths = [0.02, 0.05, 0.1, 0.15]
decays     = [0.5, 0.6, 0.7, 0.9]

print(f"{'bandwidth':>12} {'decay':>8} {'|lam|_max':>12} {'estable':>8}")
print('-' * 48)
for bw in bandwidths:
    for dc in decays:
        P = build_integral_matrix(domain, gaussian_integral_kernel,
                                  bandwidth=bw, decay=dc)
        lmax = np.max(np.abs(np.linalg.eigvals(P)))
        ok = 'OK' if lmax < 1 else 'INESTABLE'
        print(f'{bw:>12.3f} {dc:>8.2f} {lmax:>12.4f} {ok:>8}')


## 4. Dependencia temporal: FAR(1) vs proceso sin memoria


In [ ]:
sim_far = FARSimulator(domain=domain, n_curves=200, Psi=Psi,
                       trend='sinusoidal', noise_std=0.5,
                       noise_type='smooth', burn_in=100, random_state=42)
X_far = sim_far.simulate()

Psi_zero = np.zeros_like(Psi)
sim_ind  = FARSimulator(domain=domain, n_curves=200, Psi=Psi_zero,
                        trend='sinusoidal', noise_std=0.5,
                        noise_type='smooth', burn_in=0, random_state=42)
X_ind = sim_ind.simulate()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, X, title in zip(axes[0], [X_far, X_ind],
                          ['FAR(1) con memoria', 'Curvas independientes']):
    im = ax.imshow(X, aspect='auto', cmap='RdBu_r',
                   extent=[0, 1, 200, 0], vmin=-3, vmax=3)
    ax.set_title(f'Heatmap: {title}')
    ax.set_xlabel('s'); ax.set_ylabel('t')
    plt.colorbar(im, ax=ax)

for ax, X, title in zip(axes[1], [X_far, X_ind], ['FAR(1)', 'Independiente']):
    idx = np.argmin(np.abs(domain.grid - 0.5))
    ser = X[:, idx]
    acf = [1.0] + [pearsonr(ser[:-l], ser[l:])[0] for l in range(1, 16)]
    ax.stem(range(len(acf)), acf, linefmt='steelblue-', markerfmt='bo',
            basefmt='k-', use_line_collection=True)
    ci = 2 / np.sqrt(len(ser))
    ax.axhline(ci,  color='gray', ls='--', lw=0.8)
    ax.axhline(-ci, color='gray', ls='--', lw=0.8)
    ax.set_title(f'ACF en s=0.5 — {title}')
    ax.set_xlabel('Lag'); ax.grid(True, alpha=0.3)

plt.suptitle('FAR(1) vs sin memoria', fontsize=14)
plt.tight_layout(); plt.show()


## 5. Validación estadística (trend='zero')

Con `trend='zero'` la media funcional debe ser ≈ 0 y la distribución
marginal en cada s debe ser gaussiana.


In [ ]:
sim_val = FARSimulator(domain=domain, n_curves=500, Psi=Psi,
                       trend='zero', noise_std=1.0, noise_type='smooth',
                       burn_in=200, random_state=123)
X_val = sim_val.simulate()

sd = sim_val.summary_stats(n_lags=15)
print('=== summary_stats ===')
for k, v in sd.items():
    if k != 'acf':
        print(f'  {k:<40s}: {v:.4f}')

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
mean_fn = X_val.mean(axis=0)
std_fn  = X_val.std(axis=0)

# Media
axes[0,0].plot(domain.grid, mean_fn, 'b-', lw=2, label='Media empírica')
axes[0,0].fill_between(domain.grid,
    mean_fn - 2*std_fn/np.sqrt(500),
    mean_fn + 2*std_fn/np.sqrt(500), alpha=0.3, label='IC 95%')
axes[0,0].axhline(0, color='r', ls='--', alpha=0.5)
axes[0,0].set_title('Media funcional (debe ser ~ 0)')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Varianza
axes[0,1].plot(domain.grid, std_fn**2, 'b-', lw=2)
axes[0,1].set_title('Varianza funcional')
axes[0,1].set_xlabel('s'); axes[0,1].grid(True, alpha=0.3)

# ACF
axes[0,2].plot(range(1, len(sd['acf'])+1), sd['acf'], 'o-')
ci = 2/np.sqrt(500)
axes[0,2].axhline(ci,  color='gray', ls='--', lw=0.8)
axes[0,2].axhline(-ci, color='gray', ls='--', lw=0.8)
axes[0,2].set_title('ACF en s=0.5'); axes[0,2].set_xlabel('Lag')
axes[0,2].grid(True, alpha=0.3)

# Marginal
idx_m = np.argmin(np.abs(domain.grid - 0.5))
axes[1,0].hist(X_val[:, idx_m], bins=30, density=True, alpha=0.7, color='steelblue')
xr = np.linspace(X_val[:,idx_m].min(), X_val[:,idx_m].max(), 200)
axes[1,0].plot(xr, stats.norm.pdf(xr, 0, X_val[:,idx_m].std()), 'r-', lw=2)
axes[1,0].set_title(f"Marginal s=0.5  p-norm={sd['normality_p_at_s05']:.3f}")

# Covarianza cruzada
i1 = np.argmin(np.abs(domain.grid - 0.3))
i2 = np.argmin(np.abs(domain.grid - 0.7))
axes[1,1].scatter(X_val[:,i1], X_val[:,i2], alpha=0.2, s=8)
rho = pearsonr(X_val[:,i1], X_val[:,i2])[0]
axes[1,1].set_title(f'Correlacion cruzada rho={rho:.3f}')
axes[1,1].set_xlabel('X_t(s=0.3)'); axes[1,1].set_ylabel('X_t(s=0.7)')

# FPCs
eigs = np.linalg.eigvalsh(np.cov(X_val.T))[::-1]
cum  = np.cumsum(eigs)/eigs.sum()*100
axes[1,2].plot(range(1, 21), cum[:20], 'bo-')
axes[1,2].axhline(90, color='crimson', ls='--', label='90%')
axes[1,2].set_title('Varianza acumulada (primeras 20 FPCs)')
axes[1,2].set_xlabel('# componente'); axes[1,2].legend()
axes[1,2].grid(True, alpha=0.3)

plt.suptitle('Validacion estadistica del proceso FAR(1)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


## 6. Sensibilidad: noise_std y bandwidth


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, noise in zip(axes, [0.2, 0.7, 1.5]):
    s = FARSimulator(domain=domain, n_curves=50, Psi=Psi,
                     trend='sinusoidal', noise_std=noise,
                     noise_type='smooth', burn_in=100, random_state=42)
    Xn = s.simulate()
    for t in range(30):
        ax.plot(domain.grid, Xn[t], alpha=0.3, lw=0.7, color='steelblue')
    ax.plot(domain.grid, Xn.mean(axis=0), color='crimson', lw=2)
    ax.set_title(f'noise_std={noise}'); ax.set_xlabel('s')
    ax.grid(True, alpha=0.3)
plt.suptitle('Efecto de noise_std', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, bw in zip(axes, [0.02, 0.05, 0.12]):
    Pb = build_integral_matrix(domain, gaussian_integral_kernel,
                               bandwidth=bw, decay=0.6)
    sb = FARSimulator(domain=domain, n_curves=50, Psi=Pb,
                      trend='zero', noise_std=0.8,
                      noise_type='smooth', burn_in=100, random_state=42)
    Xb = sb.simulate()
    for t in range(30):
        ax.plot(domain.grid, Xb[t], alpha=0.3, lw=0.7, color='darkorange')
    ax.set_title(f'bandwidth={bw}  |lam|={sb._max_eigenvalue:.3f}')
    ax.set_xlabel('s'); ax.grid(True, alpha=0.3)
plt.suptitle('Efecto del bandwidth del operador', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


## 7. Nuevos métodos: summary_stats() y plot_acf()


In [ ]:
sim_demo = FARSimulator(domain=domain, n_curves=300, Psi=Psi,
                        trend='bump', noise_std=0.6, noise_type='smooth',
                        burn_in=100, random_state=7)
_ = sim_demo.simulate()

st = sim_demo.summary_stats()
print(f"ACF lag-1 : {st['acf_lag1']:.4f}")
print(f"|lam|_max : {st['max_eigenvalue_operator']:.4f}")
print(f"p-norm    : {st['normality_p_at_s05']:.4f}")

sim_demo.plot_acf(s_points=[0.1, 0.3, 0.5, 0.7, 0.9],
                  title='ACF funcional — trend=bump')
sim_demo.plot_heatmap(title='Heatmap FAR(1) — trend=bump')
sim_demo.plot_curves(n_show=50, title='Curvas FAR(1) — trend=bump')
